In [ ]:
import os
import csv

# get the current working directory
current_dir = os.getcwd()
print(f"current work directory: {current_dir}")

# if the current working directory not ends with "notebooks", change it to the parent directory
if not current_dir.endswith("notebooks"):

    workspace_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    os.chdir(workspace_root)

    print(f"changed root work directory to: {workspace_root}")

from modules.acf.external_id_api import ApiExternalId  # noqa: E402

CONFIG_ID = "CUSTOM_TEST"
# files
custom_mapping_path = os.path.join(workspace_root, "custom", "CUSTUM_MAPPING.csv")


def load_csv(file_path, delimiter=","):
    with open(file_path, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file, delimiter=delimiter)
        return list(reader)


# Main Logic
custom_mapping_data = load_csv(custom_mapping_path, delimiter=";")

# new dict to store the equipment and its corresponding acf equipment id
equipment_dict = {}

api_external_id = ApiExternalId(CONFIG_ID)

try:
    for custom_row in custom_mapping_data:
        # read the equipment from custom_row["PAI_Equipment"]
        equipment = custom_row["PAI_Equipment"]
        # check if the equipment is in the equipment dict already
        if equipment not in equipment_dict:
            # read the ACF equipment id from api
            external_data = api_external_id.get_external_data(filter_str=f"externalId eq '{equipment}'")
            if len(external_data) == 0:
                print(f"Error: No ACF Equipment ID found for equipment: {equipment}")
                continue
            elif len(external_data) == 1:
                print(
                    f"Found ACF Equipment ID {external_data[0]["ainObjectId"]} for equipment: {equipment}"
                )
                equipment_dict[equipment] = external_data[0]["ainObjectId"]

    # finally print just the values og the equipment dict as a list
    print("Equipment IDs:")
    for equipment in equipment_dict.values():
        print(equipment)
except KeyError as e:
    print(f"Error: Column missing in CSV: {e}")
except Exception as e:
    print(f"Unknown error occured: {e}")